# Notebook 02 — Feature Engineering

**Goal:** Engineer 25+ predictive features from the cleaned Telco Churn dataset.  
We add 7 new domain-derived features, apply one-hot encoding to all categoricals,
and standardize numeric columns.

**Input:** `data/processed/telco_churn_cleaned.csv`  
**Output:** engineered DataFrame persisted in memory for modeling

In [ ]:
import sys
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_processed_data
from src.feature_engineering import (
    add_tenure_group, add_avg_monthly_spend, add_is_high_value,
    add_num_services, add_is_senior_alone, add_contract_risk_score,
    add_auto_pay_flag, encode_categoricals, scale_numerics, build_features
)

sns.set_theme(style='whitegrid', palette='muted')
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGURES / name, dpi=150, bbox_inches='tight')
    plt.show()

print('Ready.')

## 1. Load Cleaned Data

In [ ]:
df = load_processed_data()
print('Shape:', df.shape)
df.head(3)

## 2. Apply Engineered Features Step-by-Step

Each feature is added incrementally so we can inspect its distribution.

### Feature 1: `tenure_group` — 4-bucket tenure cohort

In [ ]:
df = add_tenure_group(df)
print('tenure_group value counts:')
print(df['tenure_group'].value_counts().sort_index())

### Feature 2: `avg_monthly_spend` = TotalCharges / (tenure + 1)

In [ ]:
df = add_avg_monthly_spend(df)
print('avg_monthly_spend stats:')
print(df['avg_monthly_spend'].describe().round(2))

### Feature 3: `is_high_value` — MonthlyCharges > $75

In [ ]:
df = add_is_high_value(df)
n_hv = df['is_high_value'].sum()
print(f'High-value customers (MonthlyCharges > $75): {n_hv:,} ({n_hv/len(df)*100:.1f}%)')

### Feature 4: `num_services` — number of subscribed add-ons (0–8)

In [ ]:
df = add_num_services(df)
print('num_services distribution:')
print(df['num_services'].value_counts().sort_index())

In [ ]:
# Visualise churn rate by number of services
svc_churn = df.groupby('num_services')['Churn'].mean().reset_index()
svc_churn['churn_rate_pct'] = svc_churn['Churn'] * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(svc_churn['num_services'], svc_churn['churn_rate_pct'], marker='o', color='#2196F3', linewidth=2)
ax.set_xlabel('Number of Services')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Number of Subscribed Services', fontweight='bold')
savefig('engineered_num_services_vs_churn.png')

### Feature 5: `is_senior_alone` — Senior with no partner

In [ ]:
df = add_is_senior_alone(df)
n_sa = df['is_senior_alone'].sum()
churn_sa = df[df['is_senior_alone']==1]['Churn'].mean() * 100
print(f'Senior-alone customers: {n_sa:,} | Churn rate in segment: {churn_sa:.1f}%')

### Feature 6: `contract_risk_score` — ordinal encoding (3=month-to-month, 1=two-year)

In [ ]:
df = add_contract_risk_score(df)
print('contract_risk_score value counts:')
print(df['contract_risk_score'].value_counts())

### Feature 7: `auto_pay_flag` — 1 if automatic payment method

In [ ]:
df = add_auto_pay_flag(df)
n_auto = df['auto_pay_flag'].sum()
churn_auto = df[df['auto_pay_flag']==1]['Churn'].mean() * 100
churn_manual = df[df['auto_pay_flag']==0]['Churn'].mean() * 100
print(f'Auto-pay customers: {n_auto:,}')
print(f'Churn rate — auto: {churn_auto:.1f}%  |  manual: {churn_manual:.1f}%')

## 3. One-Hot Encode Categoricals & Standardize Numerics

In [ ]:
df_encoded = encode_categoricals(df)
print('After encoding — shape:', df_encoded.shape)
print('Columns:', df_encoded.columns.tolist())

In [ ]:
df_scaled, scaler = scale_numerics(df_encoded)

feature_cols = [c for c in df_scaled.columns if c != 'Churn']
print(f'\n=== FINAL FEATURE COUNT: {len(feature_cols)} features ===')
print('\nAll features:')
for i, f in enumerate(sorted(feature_cols), 1):
    print(f'  {i:2d}. {f}')

## 4. Feature Correlation with Target

In [ ]:
corr_with_target = df_scaled.corr()['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
top20 = corr_with_target.head(20)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#FF9800' if v > 0 else '#2196F3' for v in top20.values]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson Correlation with Churn')
ax.set_title('Top 20 Features by Correlation with Churn', fontweight='bold', fontsize=13)
savefig('feature_correlation_with_churn.png')

## 5. Persist Engineered Dataset for Modeling

In [ ]:
import joblib

out_path = ROOT / 'data' / 'processed' / 'telco_churn_features.csv'
df_scaled.to_csv(out_path, index=False)
print(f'Engineered dataset saved: {out_path}')

scaler_path = ROOT / 'models' / 'scaler.pkl'
scaler_path.parent.mkdir(exist_ok=True)
joblib.dump(scaler, scaler_path)
print(f'Scaler saved: {scaler_path}')

print(f'\nFinal dataset shape: {df_scaled.shape}')
print(f'Feature count (excl. target): {len(feature_cols)}')
assert len(feature_cols) >= 25, f'Expected 25+ features, got {len(feature_cols)}'
print('ASSERTION PASSED: 25+ features confirmed.')